<a href="https://colab.research.google.com/github/charleskwakye/be-insurance-ai/blob/main/ml_belgium.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning on "Belgian motor third-part liability dataset" from a rda dataset

### DATA COLLECTION
Install `pyreadr` library, which is capable of reading R data files (.rda), as it is not part of the standard Python distribution. Then download and load the `.rda` file from `https://github.com/dutangc/CASdatasets/blob/master/data/beMTPL16.rda`, extract the DataFrame, and display its head.

In [ ]:
!pip install pyreadr
print("pyreadr installed successfully.")

Installing pyreadr...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.3/418.3 kB 10.7 MB/s eta 0:00:00
pyreadr installed successfully.



The next step is to download the `.rda` file from the its GitHub URL  using the `requests` library into to the local environment and extract the DataFrame. Then, I will display the head of the extracted DataFrame to verify the data.




In [ ]:
import requests
import os
import pyreadr


# Direct raw URL of the .rda file on GitHub
raw_url = 'https://github.com/dutangc/CASdatasets/raw/master/data/beMTPL16.rda'

# Define the local filename dynamically from the URL
file_name = os.path.basename(raw_url)

print(f"Attempting to download '{file_name}' and load it...")

try:
    # --- Download the .rda file ---
    response = requests.get(raw_url)
    response.raise_for_status() # Raise an exception for HTTP errors

    with open(file_name, 'wb') as f:
        f.write(response.content)
    print(f"'{file_name}' downloaded successfully.")

    # --- Read the .rda file with pyreadr ---
    result = pyreadr.read_r(file_name)

    # Extract the 'beMTPL16' DataFrame directly
    df_beMTPL16 = result['beMTPL16']
    print("DataFrame 'beMTPL16' extracted successfully.")
    print("Displaying the head of the DataFrame:")
    display(df_beMTPL16.head())

except KeyError:
    print("Error: 'beMTPL16' DataFrame not found in the .rda file. Available keys are:" + str(list(result.keys())))
except requests.exceptions.RequestException as e:
    print(f"Error downloading the file: {e}")
except Exception as e:
    print(f"An error occurred during file operations: {e}")

Attempting to download 'beMTPL16.rda' and load it...
'beMTPL16.rda' downloaded successfully.
DataFrame 'beMTPL16' extracted successfully.
Displaying the head of the DataFrame:


,insurance_contract,policy_year,exposure,insured_birth_year,vehicle_age,policy_holder_age,driver_license_age,vehicle_brand,vehicle_model,mileage,vehicle_power,catalog_value,claim_value,number_of_liability_claims,number_of_bodily_injury_liability_claims,claim_time,claim_responsibility_rate,driving_training_label,signal
0,C1,1,0.386301,1945,10,9,40,MERCEDES,ME-1245,30000,75,983732,2,0,0,00:00,0,No,0
1,C2,1,0.493151,1941,4,25,24,VOLKSWAGEN,VO-2461,30000,55,510562,8,0,0,07:45,0,No,0
2,C3,1,0.290411,1944,0,2,39,AUDI,AU-967,30000,120,1934768,10,0,0,00:00,0,No,0
3,C4,1,0.336986,1948,1,14,37,LANCIA,LA-2346,30000,51,536755,13,0,0,18:50,0,No,0
4,C5,1,0.219178,1928,3,7,59,CITROEN,CI-1258,30000,54,446725,14,0,0,00:00,100,No,0


Display the head of the extracted DataFrame to verify the data.

In [ ]:
display(df_beMTPL16.head())

,insurance_contract,policy_year,exposure,insured_birth_year,vehicle_age,policy_holder_age,driver_license_age,vehicle_brand,vehicle_model,mileage,vehicle_power,catalog_value,claim_value,number_of_liability_claims,number_of_bodily_injury_liability_claims,claim_time,claim_responsibility_rate,driving_training_label,signal
0,C1,1,0.386301,1945,10,9,40,MERCEDES,ME-1245,30000,75,983732,2,0,0,00:00,0,No,0
1,C2,1,0.493151,1941,4,25,24,VOLKSWAGEN,VO-2461,30000,55,510562,8,0,0,07:45,0,No,0
2,C3,1,0.290411,1944,0,2,39,AUDI,AU-967,30000,120,1934768,10,0,0,00:00,0,No,0
3,C4,1,0.336986,1948,1,14,37,LANCIA,LA-2346,30000,51,536755,13,0,0,18:50,0,No,0
4,C5,1,0.219178,1928,3,7,59,CITROEN,CI-1258,30000,54,446725,14,0,0,00:00,100,No,0


## 1. Data Validation
## Schema
| Column Name                                 | Type     | Description                                                         |
|---------------------------------------------|----------|---------------------------------------------------------------------|
| insurance_contract                           | numeric  | Unique identifier for the contract                                 |
| policy_year                                  | numeric  | Year of study/observation for the insured person                   |
| exposure                                     | numeric  | Exposure duration in years                                          |
| insured_year_birth                           | numeric  | Insured's year of birth                                             |
| vehicle_age                                  | numeric  | Age of the vehicle in years                                         |
| policy_holder_age                            | numeric  | Seniority of the insured at the insurance agency                    |
| driver_license_age                           | numeric  | Age of the driver's licence                                         |
| vehicle_brand                                | factor   | Brand of the vehicle                                                |
| vehicle_model                                | factor   | Model of the vehicle                                                |
| mileage                                      | numeric  | Mileage of the vehicle                                              |
| vehicle_power                                | numeric  | Power value of the vehicle                                          |
| catalog_value                                | numeric  | Catalog value of the vehicle                                        |
| claim_value                                  | numeric  | Value of the claim                                                  |
| number_of_liability_claims                   | numeric  | Number of liability claims                                          |
| number_of_bodily_injury_liability_claims     | numeric  | Number of bodily injury liability claims                            |
| claim_time                                   | factor   | Time (within a day) of the accident                                 |
| claim_responsibility_rate                    | numeric  | Responsibility rate (0–100%)                                        |
| driving_training_label                       | factor   | Indicator for driving training program                              |
| signal                                       | numeric  | Warning indicator (1 = warning, 0 = no warning)                     |


Perform initial data validation by checking data types, non-null counts, and summary statistics of the `df_beMTPL16` DataFrame.


**Info()**: The `info()` method provides a concise summary of a DataFrame, including the number of entries, the number of columns, the data type of each column, the number of non-null values, and memory usage. This is crucial for identifying missing data and incorrect data types right away.

**Describe()**: The `describe()` method generates descriptive statistics that summarize the central tendency, dispersion, and shape of a dataset's distribution, excluding `NaN` values. This helps in understanding the distribution and potential outliers in numerical columns.

In [ ]:
print("DataFrame Info (Data Types and Non-Null Counts):")
df_beMTPL16.info()

DataFrame Info (Data Types and Non-Null Counts):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70791 entries, 0 to 70790
Data columns (total 19 columns):
 #   Column                                    Non-Null Count  Dtype   
---  ------                                    --------------  -----   
 0   insurance_contract                        70791 non-null  category
 1   policy_year                               70791 non-null  int32   
 2   exposure                                  70791 non-null  float64 
 3   insured_birth_year                        70791 non-null  int32   
 4   vehicle_age                               70791 non-null  int32   
 5   policy_holder_age                         70791 non-null  int32   
 6   driver_license_age                        70791 non-null  int32   
 7   vehicle_brand                             70791 non-null  category
 8   vehicle_model                             70791 non-null  category
 9   mileage                                   707

### Checking for Null Values

First, let's confirm the count of null values across all columns. This re-verifies the output from `df.info()`.

In [ ]:
print("Count of Null Values per Column:")
display(df_beMTPL16.isnull().sum())

Count of Null Values per Column:


,0
insurance_contract,0
policy_year,0
exposure,0
insured_birth_year,0
vehicle_age,0
policy_holder_age,0
driver_license_age,0
vehicle_brand,0
vehicle_model,0
mileage,0


##### Calculate Missing Values as Percentage and Check for Zero Values in Numerical Columns

Next, we'll check for zero values in the numerical columns. In some datasets, a value of 0 might represent a missing or unrecorded entry, similar to a null value. We will only check numerical columns for zeros.

In [ ]:
import pandas as pd
zero_counts = {}
for col in num_col:
    zero_counts[col] = (df_beMTPL16[col] == 0).sum()

zero_counts_df = pd.DataFrame(zero_counts.items(), columns=['Column', 'Zero Count'])
display(zero_counts_df[zero_counts_df['Zero Count'] > 0].sort_values(by='Zero Count', ascending=False))

print("\nColumns with zero values:")
for col, count in zero_counts.items():
    if count > 0:
        print(f"  {col}: {count} zeros ({count/len(df_beMTPL16)*100:.2f}% of total rows)")

,Column,Zero Count
13,signal,70746
11,number_of_bodily_injury_liability_claims,69381
10,number_of_liability_claims,46080
12,claim_responsibility_rate,36017
8,catalog_value,21844
3,vehicle_age,3304
4,policy_holder_age,2331
5,driver_license_age,3



Columns with zero values:
  vehicle_age: 3304 zeros (4.67% of total rows)
  policy_holder_age: 2331 zeros (3.29% of total rows)
  driver_license_age: 3 zeros (0.00% of total rows)
  catalog_value: 21844 zeros (30.86% of total rows)
  number_of_liability_claims: 46080 zeros (65.09% of total rows)
  number_of_bodily_injury_liability_claims: 69381 zeros (98.01% of total rows)
  claim_responsibility_rate: 36017 zeros (50.88% of total rows)
  signal: 70746 zeros (99.94% of total rows)


#### 1. Columns to Drop

These columns should be removed from your dataset before training for the reasons specified.

| Column Name | Reason to Drop |
| :--- | :--- |
| `insurance_contract` | Unique identifier, no predictive value. |
| `insured_year_birth` | Redundant (you already have `policy_holder_age`). |
| `exposure` | Irrelevant for a severity-only model (it's for frequency). |
| `number_of_liability_claims` | **Data Leakage** (This is an *outcome* of the claim). |
| `number_of_bodily_injury_liability_claims` | **Data Leakage** (This is a *component* of the claim value). |
| `claim_responsibility_rate` | **Data Leakage** (This is determined *during* claim adjustment). |
| `signal` | **Near-Zero-Variance** (99.94% zero, no predictive power). |



#### 2. Target Label & Features to Keep

This is the set of columns you will use to build your model.

#### Target Label (Y)

This is the single column you are trying to predict.

| Column Name | Role | Notes |
| :--- | :--- | :--- |
| `claim_value` | **Target (Label)** | This is the value your model will predict. Remember to filter your data for `claim_value > 0` before training. |

#### Features (X)

These are the columns your model will use to make its predictions.

| Column Name | Role | Notes |
| :--- | :--- | :--- |
| `policy_year` | Feature | Good for tracking trends/inflation over time. |
| `vehicle_age` | Feature | Keep as-is. The zeros (4.67%) likely mean "new car." |
| `policy_holder_age` | Feature | **Must clean:** The zeros (3.29%) are missing data and should be imputed. |
| `driver_license_age` | Feature | **Must clean:** The 3 zero-rows are missing data. |
| `vehicle_brand` | Feature | Categorical. Will require encoding. |
| `vehicle_model` | Feature | Categorical. May have many values; consider grouping rare models. |
| `mileage` | Feature | Numeric feature. |
| `vehicle_power` | Feature | Numeric feature. |
| `catalog_value` | Feature | **Must clean:** The zeros (30.86%) are missing data and must be imputed. |
| `claim_time` | Feature | Categorical (time of day). Will require encoding. |
| `driving_training_label` | Feature | Categorical. Will require encoding. |


Identify Column Data Types

In [ ]:
cat_col = [col for col in df_beMTPL16.columns if df_beMTPL16[col].dtype == 'category']
num_col = [col for col in df_beMTPL16.columns if df_beMTPL16[col].dtype != 'category']

print('Categorical columns:', cat_col)
print('Numerical columns:', num_col)

Categorical columns: ['insurance_contract', 'vehicle_brand', 'vehicle_model', 'claim_time', 'driving_training_label']
Numerical columns: ['policy_year', 'exposure', 'insured_birth_year', 'vehicle_age', 'policy_holder_age', 'driver_license_age', 'mileage', 'vehicle_power', 'catalog_value', 'claim_value', 'number_of_liability_claims', 'number_of_bodily_injury_liability_claims', 'claim_responsibility_rate', 'signal']


In [ ]:
df_beMTPL16[cat_col].nunique()

,0
insurance_contract,58723
vehicle_brand,66
vehicle_model,1038
claim_time,1032
driving_training_label,2


In [ ]:
print("\nDescriptive Statistics for Numerical Columns:")
display(df_beMTPL16.describe())


Descriptive Statistics for Numerical Columns:


,policy_year,exposure,insured_birth_year,vehicle_age,policy_holder_age,driver_license_age,mileage,vehicle_power,catalog_value,claim_value,number_of_liability_claims,number_of_bodily_injury_liability_claims,claim_responsibility_rate,signal
count,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000,7.079100e+04,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000
mean,2.503934,0.437601,1941.394231,6.163849,9.651679,38.196875,28327.824158,77.962382,5.823853e+05,80780.786371,0.349070,0.019918,48.424362,0.000636
std,1.108830,0.180683,6.960452,4.803108,6.789322,9.549636,5711.127945,29.633158,5.460070e+05,45999.887220,0.476679,0.139719,49.628070,0.025205
min,1.000000,0.200000,1911.000000,0.000000,0.000000,0.000000,2500.000000,30.000000,0.000000e+00,2.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,0.287671,1936.000000,2.000000,4.000000,34.000000,30000.000000,55.000000,0.000000e+00,41491.500000,0.000000,0.000000,0.000000,0.000000
50%,3.000000,0.400000,1943.000000,5.000000,9.000000,41.000000,30000.000000,74.000000,5.506000e+05,82430.000000,0.000000,0.000000,0.000000,0.000000
75%,3.000000,0.556164,1947.000000,9.000000,14.000000,43.000000,30000.000000,92.000000,8.677665e+05,120472.000000,1.000000,0.000000,100.000000,0.000000
max,4.000000,1.000000,1952.000000,60.000000,30.000000,60.000000,30000.000000,487.000000,7.234528e+06,169694.000000,1.000000,1.000000,100.000000,1.000000


In [ ]:
display(df_beMTPL16.head(100))

,insurance_contract,policy_year,exposure,insured_birth_year,vehicle_age,policy_holder_age,driver_license_age,vehicle_brand,vehicle_model,mileage,vehicle_power,catalog_value,claim_value,number_of_liability_claims,number_of_bodily_injury_liability_claims,claim_time,claim_responsibility_rate,driving_training_label,signal
0,C1,1,0.386301,1945,10,9,40,MERCEDES,ME-1245,30000,75,983732,2,0,0,00:00,0,No,0
1,C2,1,0.493151,1941,4,25,24,VOLKSWAGEN,VO-2461,30000,55,510562,8,0,0,07:45,0,No,0
2,C3,1,0.290411,1944,0,2,39,AUDI,AU-967,30000,120,1934768,10,0,0,00:00,0,No,0
3,C4,1,0.336986,1948,1,14,37,LANCIA,LA-2346,30000,51,536755,13,0,0,18:50,0,No,0
4,C5,1,0.219178,1928,3,7,59,CITROEN,CI-1258,30000,54,446725,14,0,0,00:00,100,No,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,C96,1,0.227397,1946,5,5,19,CITROEN,CI-3212,30000,80,0,1455,0,0,00:00,0,No,0
96,C97,1,0.230137,1949,8,8,36,TOYOTA,TO-1471,30000,66,661089,1456,1,0,08:45,100,No,0
97,C98,1,0.213699,1936,12,10,44,MERCEDES,ME-1429,30000,160,0,1457,1,1,00:00,100,No,0
98,C99,1,0.232877,1942,1,10,42,HYUNDAI,HY-1979,30000,80,673411,1459,0,0,00:00,0,No,0
